# PV Systematic v4 — Full Pipeline + ZoLa + QA + Auto-Export
Run Stages 1–10.

### Imports and Config


In [7]:
import os
import requests
import geopandas as gpd
import pandas as pd
import folium
from shapely.geometry import Polygon, box
import subprocess
from folium.plugins import MarkerCluster
#current working file 5/11 

## Stage 1: Define Bounding Box, Fetch + Save Footprints


In [8]:
# Fetch 6 story walkups from PLUTO dataset, clean and write to csv and geojson 
subprocess.run(["python3", "scripts/fetch_selection.py"], check=True)

Wrote all_walkups_6story.csv and all_walkups_6story.geojson


CompletedProcess(args=['python3', 'scripts/fetch_selection.py'], returncode=0)

In [9]:
#Bounding box coordinates 
N_LAT = 40.7358876957036
S_LAT = 40.71195687391828
W_LON = -74.00188877257987
E_LON = -73.9721415592451

# NYC building footprints
FOOTPRINTS_URL = "https://data.cityofnewyork.us/resource/5zhs-2jue.geojson" 
#Address doesnt exist here, therefore DNE in footprints.geojson therefore DNE in footprints_gdf. Doesnt get preserved in the pv_from_footprints.py script because i dont think its passed in there to begin with. We have to get it from somewhere else. Maybe we take the all_walkups_6story.csv file that has the address and bbl and then merge it with the building footprints data using the bbl as the key? 

#Search Parameters 
params = {
    "$select": "*",
    "$where": f"within_box(the_geom,{S_LAT},{W_LON},{N_LAT},{E_LON})",
    "$limit": 50000,
}

# API is  called to fetch the footprints data for the specified bounding box AOI. Store bounding box in a variable, we use this later when building the map. The parameters include a spatial query to filter the data within the defined box and a limit on the number of records returned.
r = requests.get(FOOTPRINTS_URL, params=params)
r.raise_for_status()


# Creating a stored bounding box for debugging  TO DO this can move down 
bbox_geom = box(W_LON, S_LAT, E_LON, N_LAT)
bbox_gdf = gpd.GeoDataFrame(geometry=[bbox_geom], crs="EPSG:4326")

#Downloads all footprint polygons inside the box AOI and saves them as a geojson file. Then we read that geojson file into a GeoDataFrame using Geopandas.We can also read it directly into geopandas, but saving it as a file first allows us to inspect the raw data if needed and also makes it easier to work with in subsequent steps.
#Name column is empty in all of building footprints. 
with open("footprints.geojson", "wb") as f:
    f.write(r.content)

# Geopandas reads the GeoJSON file so that we can use them for further analysis and visualization. These libraries help us arrange/format spatial data, which requires a different format, so we can easily  index/use it. 
# footprints_gdf is now a geodataframe variable that contains all the geometries of the building footprints. walkups_attrs is a geodataframe variable that contains the attributes for the walkups. Main variables in this workflow.  

footprints_gdf = gpd.read_file("footprints.geojson")
footprints_gdf = footprints_gdf.rename(columns={"base_bbl": "bbl"})


walkups_attrs = gpd.read_file("all_walkups_6story.geojson")
walkups_csv = pd.read_csv("all_walkups_6story.csv")

#TO DO - clean walkups by dropping columns we don't need 


print("Footprints:", len(footprints_gdf))
print("Walkups and attributes:", len(walkups_attrs))
print("Walkups and attributes CSV:", len(walkups_csv))

#print(footprints_gdf.head(10))
#print(walkups_attrs.head(10))


# print first ten bbls 
#print(walkups_attr["bbl"].head(10))
#print(footprints_gdf["bbl"].head(10))  



Footprints: 7386
Walkups and attributes: 809
Walkups and attributes CSV: 809


## Clip building footprints to Polygon AOI 

In [10]:
# Defining polygon coordinates
polygon_coords = [
    (-73.99075, 40.73474), #A: 14th & Broadway 
    ( -73.97209, 40.72689), #B: 14th and FDR 
    (-73.9777,  40.71314), #C: Grand & FDR 
    (-73.98249,  40.71468), #D: Grand and E Broadway  
    (-73.99411, 40.71371), # E : E broadway and Manhattan Bridge  
    (-73.99484, 40.7158), #F: Canal and Manhattan bridge
    (-73.9986, 40.71711),  # G: Canal and Mulberry  
    (-74.00187, 40.71939),  # H: Canal and Broadway  
    (-73.99143, 40.73179),  # I: Broadway and 10th 
     (-73.99075, 40.73474)  #A: 14th & Broadway, close polygon
]


#Using the Shapely library, Create polygon for clipping.
# GDF is used to create the polygon geometry with the appropriate coordinate reference system (CRS). 


polygon = Polygon(polygon_coords)
polygon_aoi = gpd.GeoDataFrame(geometry=[polygon], crs="EPSG:4326")

#Clip footprints_gdf to polygon and re write the geojson file with the clipped footprints. 
footprints_gdf = gpd.clip(footprints_gdf, polygon_aoi) 
footprints_gdf.to_file("footprints.geojson", driver="GeoJSON") 

bbl_list = footprints_gdf["bbl"].dropna().unique()
print(len(bbl_list))


# Use clipped footprints_gdf to filter the walkups_attrs to only include rows where the bbl is in the list of bbls from the clipped footprints_gdf. This ensures that we are only working with the walkups that are within our defined polygon area.
walkups_clipped = walkups_attrs[
    walkups_attrs["bbl"].isin(bbl_list)
].copy()

walkups_clipped.to_file("all_walkups_6story_poly.geojson", index=False)
walkups_clipped.to_csv("all_walkups_6story_poly.csv", index=False) 

print("Clipped walkups:", len(walkups_clipped))



# print(footprints_gdf.head(10))
# print(walkups_attrs.head(10))


4250
Clipped walkups: 626


### Stage 2: Generate PV Data from Footprints

In [11]:
#TO DO test the other pv scripts once we get the new base layer working on the mpa. we need to call the walkups geojson not footprints geojson for the pv calculations . 


subprocess.run(["python3", "scripts/pv_from_footprints.py"], check=True)

/Users/arfachowdhary/Ecolibrium/scripts/pv_from_footprints.py:142: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  centroids["geometry"] = centroids.geometry.centroid


Buildings processed: 657
PV footprints: 729
Centroids: 729
Pipeline complete ✔


CompletedProcess(args=['python3', 'scripts/pv_from_footprints.py'], returncode=0)

### Stage 3: Load PV Footprints and Clip to AOI Polygon 


In [12]:

print(len(footprints_gdf))
print(len(walkups_clipped))


# Loading calculated solar data files created in pv_from_footprints.py as geodataframe for further analysis and manipulation.
pv_fp = gpd.read_file(
    "pv_canopy_footprints.geojson"
)
pv_centroids = gpd.read_file(
    "pv_canopy_centroids.geojson"
)



# Clean data by selecting only the relevant columns for the PV footprints. Focusing on the key attributes needed for analysis and visualization, such as the building identifier (bbl), the estimated DC power of the PV system (pv_kw_dc), the annual energy production (annual_kwh), and the geometry of the PV canopy. By filtering out unnecessary columns, we can streamline our data processing and make it easier to work with the essential information. 

pv_fp = pv_fp.rename(columns={
    "pv_kw_dc": "pvkw",
    "annual_kwh": "annual_energy",
    "psh_daily_kwh": "pshs"
})

pv_centroids = pv_centroids.rename(columns={
    "pv_kw_dc": "pvkw",
    "annual_kwh": "annual_energy",
    "psh_daily_kwh": "pshs"
})


4780
626


### Stage 4: Data Cleaning For Folium

In [13]:
#This merge  brings the address information from the walkups_clipped layer into the footprints_gdf building footrints layer The merge is performed on the "bbl" column, which is a common identifier between the two datasets. The "how='left'" parameter ensures that we keep all records from the footprints_gdf layer and only bring in matching addresses from the walkups_clipped layer. If there are any discrepancies in the bbls between the layers, we might end up with some missing addresses in the footprints_gdf layer, which is something to be mindful of when analyzing the data later on. i just realized maybe we dont need to do tis cus we arent even mapping footprints_gdf. 
footprints_gdf = footprints_gdf.drop(
    columns=["address", "address_x", "address_y"],
    errors="ignore"
)


walkups_mappable = footprints_gdf.merge(
    walkups_clipped.drop(columns="geometry", errors="ignore"),
    on="bbl",
    how="inner"
)

print("WALKUPS CLIPPED COLUMNS :",walkups_clipped.columns.tolist()) #yes address 
print("WALKUPS MAPPABLE COLUMNS :",walkups_mappable.columns.tolist()) #yes address 

print("footprints_gdf COLUMNS :",footprints_gdf.columns.tolist()) #no address 
print("PV_FP COLUMNS :",pv_fp.columns.tolist())#yes address 

# print("first ten foorpint gdf : ", footprints_gdf["footprint_address"].head(10))

historic_gdf = gpd.read_file("historic_districts.geojson")




#Cleaning each layer's data for folium by converting every non-geometry field to string. Folium sometimes  crashes when GeoJSON properties contain NumPy value, mixed types, nulls, non-serializable objects. This ensures the map can safely render. 
def make_json_safe(gdf):
    gdf = gdf.copy()
    for col in gdf.columns:
        if col != "geometry": gdf[col] = gdf[col].fillna("").astype(str)
    return gdf



bbox_safe = make_json_safe(bbox_gdf)
polygon_aoi_safe = make_json_safe(polygon_aoi) 

buildings_safe = make_json_safe(footprints_gdf)
walkups_safe = make_json_safe(walkups_mappable)
pv_fp_safe = make_json_safe(pv_fp)
pv_centroids_safe = make_json_safe(pv_centroids)

historic_safe = make_json_safe(historic_gdf)

print("Buildings BBL count:", buildings_safe["bbl"].nunique())
print("Walkups first ten addresses:", walkups_safe.address[1:11])
print("Walkups BBL count:", walkups_safe["bbl"].nunique())
print("PV FP BBL count:", pv_fp_safe["bbl"].nunique())
print("Centroid BBL count:", pv_centroids_safe["bbl"].nunique())



WALKUPS CLIPPED COLUMNS : ['healthcenterdistrict', 'bldgdepth', 'bct2020', 'zipcode', 'numbldgs', 'zmcode', 'spdist1', 'lottype', 'latitude', 'proxcode', 'xcoord', 'otherarea', 'splitzone', 'retailarea', 'healtharea', 'pfirm15_flag', 'yearbuilt', 'lotfront', 'sanitboro', 'cb2010', 'landuse', 'assessland', 'garagearea', 'cd', 'zoningdate', 'ltdheight', 'appbbl', 'sanborn', 'commfar', 'dcpedited', 'residfar', 'builtfar', 'bldgclass', 'bbl', 'yearalter2', 'irrlotcode', 'edesigdate', 'tract2010', 'council', 'schooldist', 'spdist2', 'easements', 'longitude', 'officearea', 'histdist', 'yearalter1', 'strgearea', 'rpaddate', 'lotarea', 'numfloors', 'version', 'ycoord', 'block', 'unitsres', 'lotdepth', 'landmkdate', 'facilfar', 'overlay1', 'spdist3', 'ownertype', 'zonedist4', 'bldgfront', 'sanitdistrict', 'exempttot', 'areasource', 'assesstot', 'firecomp', 'edesignum', 'lot', 'zonedist1', 'address', 'policeprct', 'ct2010', 'ext', 'borocode', 'notes', 'sanitsub', 'taxmap', 'polidate', 'ownername

### Stage 5: Build Folium Map

In [14]:
import datetime
import pandas as pd

# --- FORMAT SOLAR DATA FIRST  ---
def format_solar(row):
    def safe(val, fmt):
        try:
            if pd.isna(val):
                return "n/a"
            return fmt.format(float(val))
        except:
            return "n/a"

    return {
        "pvkw_fmt": safe(row.get("pvkw"), "{:.1f} kW"),
        "psh_fmt": safe(row.get("pshs"), "{:.2f}"),
        "annual_fmt": safe(row.get("annual_energy"), "{:,.0f} kWh")
    }

solar_fmt = pv_fp_safe.apply(format_solar, axis=1, result_type="expand")

pv_fp_safe["pvkw_fmt"] = solar_fmt["pvkw_fmt"]
pv_fp_safe["psh_fmt"] = solar_fmt["psh_fmt"]
pv_fp_safe["annual_fmt"] = solar_fmt["annual_fmt"]


#stringify timestamps in all layers to ensure folium can render them without issues. Folium can have trouble rendering GeoJSON properties that contain datetime objects, so we convert any datetime fields to strings in ISO format. This way, we preserve the information while ensuring compatibility with the mapping library.
def stringify_timestamps(gdf):
    gdf = gdf.copy()
    for col in gdf.columns:
        if pd.api.types.is_datetime64_any_dtype(gdf[col]):
            gdf[col] = gdf[col].astype(str)
        elif gdf[col].dtype == object:
            gdf[col] = gdf[col].apply(
                lambda v: v.isoformat() if isinstance(v, (pd.Timestamp, datetime.datetime, datetime.date)) else v
            )
    return gdf

buildings_safe = stringify_timestamps(buildings_safe)
walkups_safe = stringify_timestamps(walkups_safe)   
pv_fp_safe = stringify_timestamps(pv_fp_safe)


#OpenStreetMap tile servers rejecting your requests and causing html blockages/popups. Trying CartoDB 
# m = folium.Map(location=[(N_LAT+S_LAT)/2, (E_LON+W_LON)/2], zoom_start=14)

m = folium.Map( 
    location=[N_LAT, E_LON],
    zoom_start=14,
    tiles="CartoDB positron"
)


def safe_tooltip_fields(footprints_gdf, desired_fields):
    return [f for f in desired_fields if f in footprints_gdf.columns]


fields = [
    "address",
    "shape_area",
    "construction_year",
    "ground_elevation",
    "shape_length",
    "height_roof",
]



walkups_fields = safe_tooltip_fields(walkups_safe, fields)
b_fields = safe_tooltip_fields(buildings_safe, fields)

pv_fields = safe_tooltip_fields(pv_fp_safe, fields)

solar_fields = [
    "pvkw_fmt",
    "psh_fmt",
    "annual_fmt"
]
#  old - solar_fields = [
#     "pv_area",
#     "pv_kw_dc",
#     "psh_daily_kwh",
#     "annual_kwh"
# ]

solar_fields = safe_tooltip_fields(pv_fp_safe, solar_fields)
combined_fields = pv_fields + solar_fields


# --- FIX: match aliases length exactly ---
solar_aliases = ["PV Size", "PSH", "Annual Energy"][:len(solar_fields)]

combined_aliases = (
    [f"Building: {f}" for f in pv_fields] +
    solar_aliases
)
# old - combined_aliases = (
#     [f"Building: {f}" for f in pv_fields] +
#     [f"Solar: {f}" for f in solar_fields]
# )


# --- Bounding Box for debugging ---
folium.GeoJson(
    bbox_safe,
    name="Bounding Box Perimeter",
    style_function=lambda x: {
        "color": "blue",
        "weight": 3,
        "fillOpacity": 0
    }
).add_to(m)


# --- Polygon AOI for debugging ---
folium.GeoJson(
    polygon_aoi_safe,
    name="Polygon AOI Perimeter",
    style_function=lambda x: {
        "color": "purple",
        "weight": 3,
        "fillOpacity": 0
    }
).add_to(m)


# --- Walkup footprints + attributes in AOI ---
folium.GeoJson(
    walkups_safe,
    name="6 Story Walkups",
    tooltip=folium.GeoJsonTooltip (
        fields=walkups_fields,
        aliases=[f"{f}: " for f in walkups_fields],
        sticky=True
    ),
    style_function=lambda x: {
        "fillColor": "#9ecae1",
        "strokeColor": "#6baed6",
        "weight": 1,
        "fillOpacity": 0.2
    }
).add_to(m)
# --- All Building Footprints in AOI ---
# folium.GeoJson(
#     buildings_safe,
#     name="All Building Footprints",
#     tooltip=folium.GeoJsonTooltip (
#         fields=b_fields,
#         aliases=[f"{f}: " for f in b_fields],
#         sticky=True
#     ),
#     style_function=lambda x: {
#         "fillColor": "#8d9ea7",
#         "strokeColor": "#546874",
#         "weight": 1,
#         "fillOpacity": 0.2
#     }
# ).add_to(m)

def pv_style(feature):
    kw = feature["properties"].get("pv_kw_dc", 0)

    try:
        kw = float(kw)
    except:
        kw = 0

    # normalize (tune max threshold for your dataset)
    intensity = min(kw / 50.0, 1.0)

    return {
        "fillColor": "#ff7f00",
        "color": "#cc5500",
        "weight": 1,
        "fillOpacity": 0.1 + 0.7 * intensity
}


# --- Individual PV Footprints  ---
folium.GeoJson(
    pv_fp_safe,
    name="PV Canopy Footprints",
    tooltip=folium.GeoJsonTooltip(
        fields=combined_fields,
        aliases=combined_aliases,
        sticky=True 
    ),
    style_function=lambda x: {
        "fillColor": "#ffb347",
        "color": "#f59f00",
        "weight": 1,
        "fillOpacity": 0.6
}
).add_to(m)

folium.GeoJson(
    pv_fp_safe,
    name="PV Canopy (modeled intensity)",
    style_function=pv_style,
    tooltip=folium.GeoJsonTooltip(fields=combined_fields)
).add_to(m)


# --- PV Centroids (clustered + toggleable) --- TO DO PV size is n/a here but not on canopy 

cluster_group = folium.FeatureGroup(name="PV Sites (Centroids)", show=True) 
cluster = MarkerCluster().add_to(cluster_group)

# 🔧 Apply SAME formatter to centroids (so formatting matches footprints)
solar_fmt_centroids = pv_centroids_safe.apply(format_solar, axis=1, result_type="expand")

pv_centroids_safe["pvkw_fmt"] = solar_fmt_centroids["pvkw_fmt"]
pv_centroids_safe["psh_fmt"] = solar_fmt_centroids["psh_fmt"]
pv_centroids_safe["annual_fmt"] = solar_fmt_centroids["annual_fmt"]
                                                      



for r in pv_centroids_safe.itertuples(index=False):
    lat, lon = r.geometry.y, r.geometry.x

    addr = getattr(r, "address", None) or "(address unavailable)"
    bbl  = getattr(r, "bbl", None) or "(BBL unavailable)"

    # ✅ use formatted values
    pvkw_txt = getattr(r, "pvkw_fmt", "n/a")
    psh_txt  = getattr(r, "psh_fmt", "n/a")
    annual_txt = getattr(r, "annual_fmt", "n/a")

    folium.Marker( 
        [lat, lon],
        tooltip=f"""
        <b>Solar Data</b><br>
        PV Size: {pvkw_txt}<br>
        PSH: {psh_txt}<br>
        Annual Energy: {annual_txt}
        """,
    ).add_to(cluster)

cluster_group.add_to(m)



folium.GeoJson(
    historic_safe,
    name="Historic Districts",
    tooltip=folium.GeoJsonTooltip(
        fields=safe_tooltip_fields(
            historic_safe,
            ["area_name", "desdate"]
        ),
        sticky=True
    ),
    style_function=lambda x: {
        "fillColor": "#d4b483",
        "color": "#8c6d46",
        "weight": 2,
        "fillOpacity": 0.25
    }
).add_to(m)





# --- Layer control + save ---
folium.LayerControl(collapsed=False).add_to(m)

import os
os.makedirs("maps", exist_ok=True)
OUT_HTML = "maps/solar_map.html"

m.save(OUT_HTML)
print("Saved", OUT_HTML)

Saved maps/solar_map.html
